# Credibility Theory — CAS Exam 5 Study Notebook

**Source:** Tse, *Non-life Insurance Mathematics* (2nd ed.), Chapters 6–9
- **Ch 6:** Classical (Limited-Fluctuation) Credibility
- **Ch 7:** Bühlmann / Bühlmann–Straub Credibility
- **Ch 8:** Bayesian Approach
- **Ch 9:** Empirical Implementation of Credibility

**Topics:**
1. Classical credibility — full credibility standard, partial credibility (square-root rule)
2. Bühlmann credibility (least-squares / greatest-accuracy)
3. Bayesian credibility
4. Empirical credibility — nonparametric estimation of EPV and VHM

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:,.4f}'.format)

---
## 1. Classical (Limited-Fluctuation) Credibility — Tse Ch 6

### Framework

The **updated prediction** for a risk group is a weighted average of the observed
experience $D$ and the manual (prior) rate $M$:

$$U = Z D + (1 - Z) M$$

where $Z \in [0, 1]$ is the **credibility factor**.

- $Z = 1$: **full credibility** — observed data used entirely
- $Z < 1$: **partial credibility** — blend data with manual rate

The classical approach asks: *How much data is needed before we give full credibility?*

### 1.1 Full Credibility Standards

Full credibility is granted when the probability that the observed loss measure
falls within $\pm 100k\%$ of the true mean is at least $1-\alpha$.

Under the **Poisson assumption for claim frequency** (with normal approximation),
the **standard for full credibility** for each loss measure is:

| Loss Measure | Standard for Full Credibility |
|---|---|
| Claim frequency | $\lambda_F = \left(\dfrac{z_{1-\alpha/2}}{k}\right)^2$ |
| Claim severity | $\lambda_F C_X^2$ (in number of claims) |
| Aggregate loss / Pure premium | $\lambda_F(1 + C_X^2)$ (in expected claims) |

where:
- $z_{1-\alpha/2}$ = standard normal critical value for coverage probability $1-\alpha$
- $k$ = accuracy parameter (e.g., 0.05 means within 5% of true mean)
- $C_X = \sigma_X / \mu_X$ = coefficient of variation of claim severity

Note: **Agg. loss standard = freq. standard + severity standard** (since $\lambda_F(1+C_X^2) = \lambda_F + \lambda_F C_X^2$)

**Common default:** $k = 0.05$, $1-\alpha = 0.90$ gives $\lambda_F = (1.645/0.05)^2 \approx 1{,}082$

In [ ]:
# ── Full Credibility Standards ─────────────────────────────────────────────

def full_credibility_freq(k, alpha):
    """Standard for full credibility for claim frequency (Poisson assumption)."""
    z = stats.norm.ppf(1 - alpha / 2)
    return (z / k) ** 2

def full_credibility_severity(k, alpha, cv_x):
    """Standard for full credibility for claim severity (number of claims)."""
    lf = full_credibility_freq(k, alpha)
    return lf * cv_x ** 2

def full_credibility_aggregate(k, alpha, cv_x):
    """Standard for full credibility for aggregate loss / pure premium."""
    lf = full_credibility_freq(k, alpha)
    return lf * (1 + cv_x ** 2)

# ── Reproduce Table 6.1: lambda_F for selected alpha and k ─────────────────
print("Full Credibility Standard λ_F for Claim Frequency")
print("=" * 55)
alphas = [0.20, 0.10, 0.05, 0.01]
ks = [0.10, 0.05, 0.01]
header = f"{'Coverage':>10} {'k=10%':>10} {'k=5%':>10} {'k=1%':>10}"
print(header)
print("-" * 45)
for alpha in alphas:
    row = f"{1-alpha:>10.0%}"
    for k in ks:
        lf = full_credibility_freq(k, alpha)
        row += f"  {lf:>8,.0f}"
    print(row)

In [ ]:
# ── Example 6.7: Full credibility for claim severity ──────────────────────
mu_x = 1_000
var_x = 2_000_000
k = 0.05
alpha = 0.01

cv_x = np.sqrt(var_x) / mu_x
lf = full_credibility_freq(k, alpha)
sev_standard = full_credibility_severity(k, alpha, cv_x)

print(f"Severity mean      : {mu_x:,}")
print(f"Severity variance  : {var_x:,}")
print(f"CV of severity (Cx): {cv_x:.4f}")
print(f"λ_F (freq standard): {lf:,.2f}")
print(f"Severity standard  : λ_F × Cx² = {lf:.2f} × {cv_x**2:.4f} = {sev_standard:,.2f}")
print()
print("Interpretation: need at least", int(np.ceil(sev_standard)),
      "claims for full severity credibility")

In [ ]:
# ── Example 6.8: Freq vs aggregate loss full credibility ──────────────────
mu_x = 25
var_x = 800
k = 0.08
alpha = 0.15   # => 1-alpha = 0.85 coverage
expected_claims = 400

cv_x = np.sqrt(var_x) / mu_x
lf = full_credibility_freq(k, alpha)
agg_standard = full_credibility_aggregate(k, alpha, cv_x)

print(f"z_{{1-α/2}} = {stats.norm.ppf(1 - alpha/2):.4f}")
print(f"λ_F (claim frequency standard)   : {lf:,.2f}")
print(f"Severity CV                       : {cv_x:.4f}")
print(f"Aggregate loss standard           : {agg_standard:,.2f}")
print()
print(f"Expected claims next period: {expected_claims}")
print(f"Full credibility for frequency?   {expected_claims} >= {lf:.0f} => {expected_claims >= lf}")
print(f"Full credibility for agg. loss?   {expected_claims} >= {agg_standard:.0f} => {expected_claims >= agg_standard}")

### 1.2 Partial Credibility — Square-Root Rule

When the data are insufficient for full credibility, the **square-root rule** gives
the partial credibility factor:

$$Z = \sqrt{\frac{\text{observed size}}{\text{standard for full credibility}}}$$

Specifically for each loss measure (assuming Poisson frequency):

| Loss Measure | Partial Credibility Factor $Z$ |
|---|---|
| Claim frequency | $Z = \sqrt{\dfrac{\lambda_N}{\lambda_F}}$ |
| Claim severity | $Z = \sqrt{\dfrac{N}{\lambda_F C_X^2}}$ |
| Aggregate loss / Pure premium | $Z = \sqrt{\dfrac{\lambda_N}{\lambda_F(1+C_X^2)}}$ |

The updated prediction is then $U = Z D + (1 - Z) M$.

**Rule:** If $Z \geq 1$ (i.e., observed ≥ standard), set $Z = 1$ (full credibility).

In [ ]:
# ── Partial Credibility: Square-Root Rule ─────────────────────────────────

def partial_Z(observed, standard):
    """Partial credibility factor: min(sqrt(observed/standard), 1.0)."""
    return min(np.sqrt(observed / standard), 1.0)

# ── Example 6.11: Partial Z for freq, severity, aggregate ─────────────────
n_claims = 896          # observed claims
mean_loss = 45
var_loss = 5_067
lambda_N = 18_600 * 0.09   # expected claims = policies × freq per policy
k = 0.10
alpha = 0.02            # 98% coverage

cv_x = np.sqrt(var_loss) / mean_loss
lf = full_credibility_freq(k, alpha)
lf_sev = full_credibility_severity(k, alpha, cv_x)
lf_agg = full_credibility_aggregate(k, alpha, cv_x)

Z_freq = partial_Z(lambda_N, lf)
Z_sev  = partial_Z(n_claims, lf_sev)
Z_agg  = partial_Z(lambda_N, lf_agg)

print(f"λ_N (expected claims)      : {lambda_N:,.0f}")
print(f"CV of claim severity       : {cv_x:.4f}")
print()
print(f"Full credibility standard  (freq):     {lf:,.2f}")
print(f"Full credibility standard  (severity): {lf_sev:,.2f}")
print(f"Full credibility standard  (aggregate):{lf_agg:,.2f}")
print()
print(f"Z (frequency)  : {Z_freq:.4f}  ({'Full' if Z_freq==1 else 'Partial'})")
print(f"Z (severity)   : {Z_sev:.4f}  ({'Full' if Z_sev==1 else 'Partial'})")
print(f"Z (aggregate)  : {Z_agg:.4f}  ({'Full' if Z_agg==1 else 'Partial'})")

In [ ]:
# ── Classical credibility update formula ──────────────────────────────────
# Suppose manual rate M = 50, observed severity D = 45
M = 50   # manual rate
D = 45   # observed mean severity

U = Z_sev * D + (1 - Z_sev) * M
print(f"Updated predicted severity: U = {Z_sev:.4f} × {D} + {1-Z_sev:.4f} × {M} = {U:.4f}")
print()
print("Summary of Classical Credibility:")
print("-" * 50)
summary = pd.DataFrame({
    'Measure': ['Frequency', 'Severity', 'Aggregate/PP'],
    'Full Cred Standard': [f'{lf:,.0f}', f'{lf_sev:,.0f}', f'{lf_agg:,.0f}'],
    'Observed Size':  [f'{lambda_N:,.0f}', f'{n_claims:,}', f'{lambda_N:,.0f}'],
    'Z': [f'{Z_freq:.4f}', f'{Z_sev:.4f}', f'{Z_agg:.4f}'],
    'Full Cred?': [Z_freq==1, Z_sev==1, Z_agg==1],
})
print(summary.to_string(index=False))

---
## 2. Bühlmann Credibility (Least-Squares / Greatest-Accuracy) — Tse Ch 7

### Framework

The Bühlmann model places the credibility problem in a rigorous statistical
framework. It finds the **linear predictor** of the next loss that **minimizes
mean squared error** (MSE).

**Setup:**
- Risk group is characterized by a parameter $\theta$ (realization of random variable $\Theta$)
- $X_1, \ldots, X_n$: iid loss observations, distribution depends on $\theta$
- Task: predict $X_{n+1}$ as $\hat{X}_{n+1} = \beta_0 + \beta_1 X_1 + \cdots + \beta_n X_n$

### Variance Decomposition

Total variance decomposes into two components:

$$\text{Var}(X) = \underbrace{E[\text{Var}(X|\Theta)]}_{\text{EPV}} + \underbrace{\text{Var}[E(X|\Theta)]}_{\text{VHM}}$$

- **EPV** = Expected Value of Process Variance = $\mu_{PV}$ = average within-group variance
- **VHM** = Variance of Hypothetical Means = $\sigma^2_{HM}$ = between-group variance

**Bühlmann credibility parameter:**
$$k = \frac{\text{EPV}}{\text{VHM}} = \frac{\mu_{PV}}{\sigma^2_{HM}}$$

Small $k$ (small EPV relative to VHM) → risk groups are very different → more weight on data.

In [ ]:
# ── Bühlmann Variance Components: Example 7.5 ─────────────────────────────
# Workers compensation: 3 risk groups with Poisson frequency and Gamma severity
# Risk group data:
#   Group | Prob | lambda | alpha | beta
groups = [
    {'prob': 0.2, 'lam': 20, 'alpha': 5, 'beta': 2},
    {'prob': 0.4, 'lam': 30, 'alpha': 4, 'beta': 3},
    {'prob': 0.4, 'lam': 40, 'alpha': 3, 'beta': 2},
]

# (a) Claim frequency N ~ Poisson(lambda)
# E(N|lambda)=lambda, Var(N|lambda)=lambda
probs = np.array([g['prob'] for g in groups])
lams  = np.array([g['lam']  for g in groups])
mu_N  = np.sum(probs * lams)          # unconditional mean
EPV_N = np.sum(probs * lams)          # E[Var(N|Lambda)] = E[Lambda] = mu_N for Poisson
VHM_N = np.sum(probs * lams**2) - mu_N**2  # Var[E(N|Lambda)]

print("(a) Claim Frequency N")
print(f"  Unconditional mean E(N)  : {mu_N:.2f}")
print(f"  EPV = E[Var(N|Λ)]        : {EPV_N:.2f}")
print(f"  VHM = Var[E(N|Λ)]        : {VHM_N:.2f}")
print(f"  Total variance           : {EPV_N + VHM_N:.2f}")
print(f"  k = EPV/VHM              : {EPV_N/VHM_N:.4f}")

# (b) Claim severity X ~ Gamma(alpha, beta)  => E(X)=alpha*beta, Var(X)=alpha*beta^2
# Probability of each severity type is weighted by expected claims per group
expected_claims = probs * lams
sev_prob = expected_claims / expected_claims.sum()

alphas = np.array([g['alpha'] for g in groups])
betas  = np.array([g['beta']  for g in groups])
mu_X_cond  = alphas * betas       # E(X|Gamma)
var_X_cond = alphas * betas**2    # Var(X|Gamma)

mu_X  = np.sum(sev_prob * mu_X_cond)
EPV_X = np.sum(sev_prob * var_X_cond)
VHM_X = np.sum(sev_prob * mu_X_cond**2) - mu_X**2

print("\n(b) Claim Severity X (Gamma)")
print(f"  Severity probabilities   : {sev_prob}")
print(f"  Unconditional mean E(X)  : {mu_X:.4f}")
print(f"  EPV = E[Var(X|Γ)]        : {EPV_X:.4f}")
print(f"  VHM = Var[E(X|Γ)]        : {VHM_X:.4f}")
print(f"  k = EPV/VHM              : {EPV_X/VHM_X:.4f}")

In [ ]:
# (c) Aggregate Loss S — compound Poisson with Gamma severity
# E(S|Θ) = lambda*alpha*beta
# Var(S|Θ) = lambda*(alpha*beta^2 + alpha^2*beta^2) = lambda*alpha*beta^2*(1+alpha)
mu_S_cond  = lams * alphas * betas
var_S_cond = lams * alphas * betas**2 * (1 + alphas)

mu_S  = np.sum(probs * mu_S_cond)
EPV_S = np.sum(probs * var_S_cond)
VHM_S = np.sum(probs * mu_S_cond**2) - mu_S**2

print("(c) Aggregate Loss S")
print(f"  Unconditional mean E(S)  : {mu_S:.2f}")
print(f"  EPV = E[Var(S|Θ)]        : {EPV_S:.2f}")
print(f"  VHM = Var[E(S|Θ)]        : {VHM_S:.2f}")
print(f"  Total variance           : {EPV_S + VHM_S:.2f}")
print(f"  k = EPV/VHM              : {EPV_S/VHM_S:.4f}")

### Bühlmann Credibility Factor and Updating Formula

The MSE-optimal linear predictor (the **Bühlmann premium**) is:

$$\hat{X}_{n+1} = Z \bar{X} + (1 - Z)\mu_X$$

where:
$$Z = \frac{n}{n+k}, \qquad k = \frac{\text{EPV}}{\text{VHM}}, \qquad \mu_X = E(X)$$

Key properties:
- $Z \to 1$ as $n \to \infty$ (more data → more weight on experience)
- $Z \to 1$ as $k \to 0$ (more distinguishable groups → more weight on experience)
- $\bar{X}$ = sample mean of $n$ observations for this risk group
- $\mu_X$ = unconditional (overall) mean = credibility complement

In [ ]:
# ── Bühlmann Credibility Factor and Updating ──────────────────────────────

def buhlmann_Z(n, EPV, VHM):
    k = EPV / VHM
    return n / (n + k), k

# ── Example 7.7: Update prediction after 1 year of experience ─────────────
# From Example 7.5: k values for N, X, S
n_periods = 1         # one year of experience
observed_N = 26       # claims observed
obs_avg_X  = 12.0     # average claim severity

# Frequency
Z_N, k_N = buhlmann_Z(n_periods, EPV_N, VHM_N)
pred_N = Z_N * observed_N + (1 - Z_N) * mu_N

# Severity (n = number of claims)
Z_X, k_X = buhlmann_Z(observed_N, EPV_X, VHM_X)
pred_X = Z_X * obs_avg_X + (1 - Z_X) * mu_X

# Aggregate loss (n = number of periods)
obs_S = observed_N * obs_avg_X   # = 312
Z_S, k_S = buhlmann_Z(n_periods, EPV_S, VHM_S)
pred_S = Z_S * obs_S + (1 - Z_S) * mu_S

print("Bühlmann Credibility Predictions (Example 7.7)")
print("=" * 55)
print(f"{'Measure':<20} {'k':>8} {'Z':>8} {'Observed':>10} {'Prior µ':>10} {'Prediction':>12}")
print("-" * 68)
print(f"{'Freq N (n=1)':<20} {k_N:>8.4f} {Z_N:>8.4f} {observed_N:>10.2f} {mu_N:>10.2f} {pred_N:>12.4f}")
print(f"{'Severity X (n=26)':<20} {k_X:>8.4f} {Z_X:>8.4f} {obs_avg_X:>10.2f} {mu_X:>10.4f} {pred_X:>12.4f}")
print(f"{'Aggregate S (n=1)':<20} {k_S:>8.4f} {Z_S:>8.4f} {obs_S:>10.2f} {mu_S:>10.2f} {pred_S:>12.4f}")

### Bühlmann–Straub Credibility

The standard Bühlmann model assumes all observations are identically distributed.
The **Bühlmann–Straub** extension allows different exposures $m_i$ per period.

Let $X_i$ = loss per unit of exposure in period $i$, with exposure $m_i$.

**Assumption:** $\text{Var}(X_i | \theta) = \sigma^2_X(\theta) / m_i$

The MSE-optimal predictor is still:
$$\hat{X}_{n+1} = Z \bar{X} + (1 - Z)\mu_X$$

but now with **exposure-weighted mean** and **exposure-weighted $Z$**:

$$\bar{X} = \frac{\sum_{i=1}^n m_i X_i}{m}, \quad m = \sum_{i=1}^n m_i, \quad Z = \frac{m}{m + k}$$

where $k = \text{EPV} / \text{VHM}$ is unchanged.

The Bühlmann model is a special case where all $m_i = 1$ (so $m = n$).

In [ ]:
# ── Bühlmann–Straub: Example 7.9 ──────────────────────────────────────────
# Binomial(2, theta) claim frequency; theta ~ Beta(alpha=1, beta=10)
# Data: 3 years with different numbers of insureds

alpha_prior = 1
beta_prior  = 10

# Beta distribution moments
E_theta      = alpha_prior / (alpha_prior + beta_prior)
Var_theta    = (alpha_prior * beta_prior) / ((alpha_prior + beta_prior)**2 * (alpha_prior + beta_prior + 1))

print(f"Prior Beta({alpha_prior}, {beta_prior}):")
print(f"  E[Θ]   = {E_theta:.4f}")
print(f"  Var[Θ] = {Var_theta:.6f}")

# Conditional moments: X_i = claims per insured ~ BN(2, theta)
# E(X_i|Θ) = 2*Θ  =>  VHM = Var[2*Θ] = 4*Var[Θ]
# Var(X_i|Θ) / m_i = 2*Θ*(1-Θ) / m_i  =>  EPV = E[2*Θ*(1-Θ)]
VHM = 4 * Var_theta
EPV = 2 * (E_theta - (Var_theta + E_theta**2))   # = 2*E[Θ(1-Θ)] = 2*(E[Θ] - E[Θ²])
k   = EPV / VHM

# Data
years = [1, 2, 3]
insureds = np.array([100, 200, 250])   # m_i
claims   = np.array([7, 13, 18])       # total claims per year
Xi       = claims / insureds           # claims per insured

m        = insureds.sum()
X_bar    = (insureds * Xi).sum() / m  # exposure-weighted mean
mu_X     = 2 * E_theta                # unconditional mean of X_i
Z        = m / (m + k)

pred_per_insured = Z * X_bar + (1 - Z) * mu_X
pred_year4       = 280 * pred_per_insured   # 280 insureds in Year 4

print(f"\nEPV = {EPV:.4f},  VHM = {VHM:.6f},  k = {k:.4f}")
print(f"m (total exposure) = {m}")
print(f"X̄ (exposure-wtd mean) = {X_bar:.4f}")
print(f"µ_X (prior mean)   = {mu_X:.4f}")
print(f"Z (credibility)    = {Z:.4f}")
print(f"\nPredicted claims per insured (Year 4): {pred_per_insured:.4f}")
print(f"Predicted total claims in Year 4 (280 insureds): {pred_year4:.2f}")

---
## 3. Bayesian Credibility — Tse Ch 8

### Framework

The Bayesian approach treats the risk parameter $\Theta$ as a **random variable**
with a **prior distribution** $f_\Theta(\theta)$.

After observing data $\mathbf{x} = (x_1, \ldots, x_n)$, the **posterior distribution** of
$\Theta$ is updated via Bayes' theorem:

$$f_{\Theta|X}(\theta | \mathbf{x}) = \frac{f_{X|\Theta}(\mathbf{x}|\theta) \cdot f_\Theta(\theta)}{f_X(\mathbf{x})} \propto f_{X|\Theta}(\mathbf{x}|\theta) \cdot f_\Theta(\theta)$$

Under **squared-error loss**, the **Bayes estimator** (Bayesian premium) is the
**posterior mean**:

$$\hat{\mu}_X(\mathbf{x}) = E[\mu_X(\Theta) | \mathbf{x}] = E[E(X|\Theta) | \mathbf{x}]$$

This is also equal to $E(X_{n+1} | \mathbf{x})$ — the conditional expectation of
the next loss given past data.

### Conjugate Distributions

Conjugate priors yield tractable posteriors of the same family:

| Prior–Likelihood pair | Prior params after $n$ obs |
|---|---|
| **Beta–Bernoulli** | $\alpha^* = \alpha + n\bar{x},\quad \beta^* = \beta + n - n\bar{x}$ |
| **Beta–Binomial** ($m$ trials) | $\alpha^* = \alpha + n\bar{x},\quad \beta^* = \beta + mn - n\bar{x}$ |
| **Gamma–Poisson** | $\alpha^* = \alpha + n\bar{x},\quad \beta^* = \beta/(n\beta+1)$ |
| **Gamma–Exponential** | $\alpha^* = \alpha + n,\quad \beta^* = \beta/(1+\beta n\bar{x})$ |

For conjugate pairs, the Bayesian premium is easy to compute from the
posterior mean (e.g., for Gamma: $E[\Lambda] = \alpha\beta$).

In [ ]:
# ── Bayesian Credibility: Discrete Example (Example 8.6) ──────────────────
# X takes values 10, 20, 30; Θ takes values 1, 2, 3
# Prior: P(Θ=1)=0.4, P(Θ=2)=0.4, P(Θ=3)=0.2
# Observed: x = (20, 20, 30) for n=3 claims

theta_vals = np.array([1, 2, 3])
prior      = np.array([0.4, 0.4, 0.2])

# Conditional distributions P(X=x | Θ=θ)
cond_dist = {
    1: {10: 0.2, 20: 0.3, 30: 0.5},
    2: {10: 0.4, 20: 0.4, 30: 0.2},
    3: {10: 0.5, 20: 0.5, 30: 0.0},
}

# Observed sample
observations = [20, 20, 30]

# Likelihood of data given each theta: product of P(x_i | theta)
likelihoods = []
for theta in theta_vals:
    lik = 1.0
    for x in observations:
        lik *= cond_dist[theta][x]
    likelihoods.append(lik)
likelihoods = np.array(likelihoods)

# Joint: P(Θ=θ, x)
joint = likelihoods * prior

# Marginal: P(x)
marginal = joint.sum()

# Posterior: P(Θ=θ | x)
posterior = joint / marginal

# Conditional means E(X|Θ=θ)
E_X_given_theta = np.array([
    sum(x * p for x, p in cond_dist[theta].items())
    for theta in theta_vals
])

# Bayesian premium = posterior mean of µ_X(Θ)
bayesian_premium = np.sum(E_X_given_theta * posterior)

print("Bayesian Credibility (Example 8.6)")
print("=" * 50)
print(f"{'Θ':>4} {'Prior':>8} {'Likelihood':>12} {'Joint':>10} {'Posterior':>12} {'E(X|Θ)':>10}")
print("-" * 58)
for i, theta in enumerate(theta_vals):
    print(f"{theta:>4} {prior[i]:>8.3f} {likelihoods[i]:>12.5f} {joint[i]:>10.5f} {posterior[i]:>12.4f} {E_X_given_theta[i]:>10.2f}")
print()
print(f"Marginal f(x) = {marginal:.4f}")
print(f"Bayesian premium = Σ E(X|Θ) × P(Θ|x) = {bayesian_premium:.4f}")

In [ ]:
# ── Gamma–Poisson Conjugate: Bayesian Premium ─────────────────────────────
# X_1,...,X_n ~ Poisson(λ); prior Λ ~ Gamma(α, β)
# Posterior: Λ | x ~ Gamma(α*, β*) where α* = α + n*x̄,  β* = β/(nβ+1)
# Bayesian premium = E[Λ | x] = α* β*
# This equals the Bühlmann credibility estimate!

def gamma_poisson_bayesian(alpha, beta, observations):
    """
    Gamma(alpha, beta) prior conjugate to Poisson likelihood.
    Returns posterior params and Bayesian premium = E[Lambda | data].
    Using parameterization where E[Gamma] = alpha*beta.
    """
    n     = len(observations)
    x_bar = np.mean(observations)
    alpha_star = alpha + n * x_bar
    beta_star  = beta / (n * beta + 1)
    bayesian_premium = alpha_star * beta_star
    return alpha_star, beta_star, bayesian_premium

# Example: alpha=4, beta=3 (prior mean = 12), 2 years with 10, 14 claims
alpha, beta = 4, 3
obs = [10, 14]

alpha_s, beta_s, prem = gamma_poisson_bayesian(alpha, beta, obs)
prior_mean = alpha * beta
x_bar = np.mean(obs)
n = len(obs)

print(f"Gamma–Poisson Conjugate")
print(f"Prior Gamma({alpha}, {beta}):  E[Λ] = {prior_mean}")
print(f"Observations: {obs}  (n={n}, x̄={x_bar:.1f})")
print()
print(f"Posterior Gamma(α*, β*) = Gamma({alpha_s:.1f}, {beta_s:.4f})")
print(f"Bayesian premium = E[Λ|data] = {alpha_s:.1f} × {beta_s:.4f} = {prem:.4f}")
print()

# Show Bühlmann equivalence: for Gamma(alpha,beta) prior + Poisson likelihood
# EPV = E[Lambda] = alpha*beta = prior_mean
# VHM = Var[Lambda] = alpha*beta^2
EPV = alpha * beta
VHM = alpha * beta**2
k   = EPV / VHM    # = 1/beta
Z   = n / (n + k)
buhlmann_pred = Z * x_bar + (1 - Z) * prior_mean

print(f"Bühlmann equivalence check:")
print(f"  EPV = E[Λ] = {EPV},  VHM = Var[Λ] = {VHM:.4f},  k = {k:.4f}")
print(f"  Z = {n}/({n} + {k:.4f}) = {Z:.4f}")
print(f"  Bühlmann = {Z:.4f} × {x_bar} + {1-Z:.4f} × {prior_mean} = {buhlmann_pred:.4f}")
print(f"  Bayesian premium = {prem:.4f}  (same — conjugate prior + LEF => Bühlmann = Bayesian)")

### Bühlmann vs. Bayesian: When Are They Equal?

When the **likelihood belongs to the linear exponential family (LEF)** and the
**prior is the natural conjugate**, the Bühlmann credibility estimate equals
the Bayesian estimate (posterior mean).

Examples of LEF likelihoods with natural conjugate priors:
- Poisson(λ) + Gamma prior
- Bernoulli(θ) or Binomial(m,θ) + Beta prior
- Exponential(λ) + Gamma prior
- Normal(µ, σ²) with known σ² + Normal prior

In general (non-conjugate or non-LEF cases), Bühlmann gives the best **linear**
approximation to the Bayesian premium.

---
## 4. Empirical Implementation of Credibility — Tse Ch 9

In practice, EPV and VHM are **unknown** and must be **estimated from data**.

### Setup: Multiple Risk Groups

Extend to $r > 1$ risk groups, each with $n_i$ observations:
- $X_{ij}$ = loss per unit of exposure in group $i$, period $j$
- $m_{ij}$ = exposure (e.g., number of insureds)

Define:
$$m_i = \sum_j m_{ij}, \quad m = \sum_i m_i, \quad \bar{X}_i = \frac{\sum_j m_{ij} X_{ij}}{m_i}, \quad \bar{X} = \frac{\sum_i m_i \bar{X}_i}{m}$$

### Nonparametric Unbiased Estimators

**EPV estimator** (unbiased):
$$\hat{\mu}_{PV} = \frac{\sum_i \sum_j m_{ij}(X_{ij} - \bar{X}_i)^2}{\sum_i (n_i - 1)}$$

**VHM estimator** (unbiased):
$$\hat{\sigma}^2_{HM} = \frac{\left[\sum_i m_i(\bar{X}_i - \bar{X})^2\right] - (r-1)\hat{\mu}_{PV}}{m - \frac{1}{m}\sum_i m_i^2}$$

If $\hat{\sigma}^2_{HM} < 0$, set it to zero (implying $Z_i = 0$, use overall mean).

### Bühlmann–Straub Credibility Predictions

$$\hat{Z}_i = \frac{m_i}{m_i + \hat{k}}, \qquad \hat{k} = \frac{\hat{\mu}_{PV}}{\hat{\sigma}^2_{HM}}$$

**Credibility predictor for group $i$:** $\hat{Z}_i \bar{X}_i + (1 - \hat{Z}_i) \bar{X}$

### Balancing Adjustment

The sum of predicted losses may not equal the sum of observed losses. To balance:

$$\hat{\mu}_X = \frac{\sum_i \hat{Z}_i \bar{X}_i}{\sum_i \hat{Z}_i}$$

Then use $\hat{\mu}_X$ in place of $\bar{X}$ as the credibility complement.

In [ ]:
# ── Empirical Credibility: Nonparametric Estimators ───────────────────────

def buhlmann_straub_empirical(groups):
    """
    Nonparametric empirical Bühlmann–Straub credibility.

    groups: list of dicts, each with:
        'exposures': list of m_ij values
        'losses':    list of X_ij values (loss per unit exposure)

    Returns: dict with EPV_hat, VHM_hat, k_hat, Z_i, predictions, balanced_predictions
    """
    r = len(groups)

    # Per-group weighted means
    X_bar_i = []
    m_i     = []
    n_i     = []
    for g in groups:
        m_ij = np.array(g['exposures'])
        X_ij = np.array(g['losses'])
        mi   = m_ij.sum()
        m_i.append(mi)
        n_i.append(len(m_ij))
        X_bar_i.append((m_ij * X_ij).sum() / mi)

    m_i     = np.array(m_i)
    n_i     = np.array(n_i)
    X_bar_i = np.array(X_bar_i)
    m       = m_i.sum()
    X_bar   = (m_i * X_bar_i).sum() / m    # overall weighted mean

    # EPV: sum of within-group weighted SSE / total (n_i - 1)
    numerator_EPV = 0.0
    for g, mi, xbi in zip(groups, m_i, X_bar_i):
        m_ij = np.array(g['exposures'])
        X_ij = np.array(g['losses'])
        numerator_EPV += (m_ij * (X_ij - xbi)**2).sum()
    denom_EPV = (n_i - 1).sum()
    EPV_hat = numerator_EPV / denom_EPV

    # VHM
    between_ss  = (m_i * (X_bar_i - X_bar)**2).sum()
    denom_VHM   = m - (m_i**2).sum() / m
    VHM_hat = (between_ss - (r - 1) * EPV_hat) / denom_VHM
    VHM_hat = max(VHM_hat, 0.0)   # truncate to 0 if negative

    k_hat   = EPV_hat / VHM_hat if VHM_hat > 0 else np.inf
    Z_i     = m_i / (m_i + k_hat) if VHM_hat > 0 else np.zeros(r)

    # Predictions without balancing
    pred    = Z_i * X_bar_i + (1 - Z_i) * X_bar

    # Balanced predictions
    mu_hat  = (Z_i * X_bar_i).sum() / Z_i.sum() if Z_i.sum() > 0 else X_bar
    pred_balanced = Z_i * X_bar_i + (1 - Z_i) * mu_hat

    return {
        'X_bar_i': X_bar_i,
        'X_bar': X_bar,
        'm_i': m_i,
        'EPV_hat': EPV_hat,
        'VHM_hat': VHM_hat,
        'k_hat': k_hat,
        'Z_i': Z_i,
        'predictions': pred,
        'mu_hat': mu_hat,
        'predictions_balanced': pred_balanced,
    }

In [ ]:
# ── Example 9.1: Workers Compensation — 3 Companies ───────────────────────
# Company A: 3 years; B and C: 4 years
# X_ij = claims per hundred workers; m_ij = hundreds of workers

groups_ex91 = [
    # Company A: Years 2, 3, 4 (Year 1 missing)
    {'name': 'A', 'exposures': [10, 11, 12], 'losses': [1.2, 0.9, 1.8]},
    # Company B: Years 1-4
    {'name': 'B', 'exposures': [5,  5,  6,  6], 'losses': [0.6, 0.8, 1.2, 1.0]},
    # Company C: Years 1-4
    {'name': 'C', 'exposures': [8,  8,  9, 10], 'losses': [0.7, 0.9, 1.3, 1.1]},
]

res = buhlmann_straub_empirical(groups_ex91)

print("Bühlmann–Straub Empirical Credibility (Example 9.1)")
print("=" * 65)
print(f"Overall weighted mean X̄ = {res['X_bar']:.4f}")
print(f"EPV hat (µ_PV)          = {res['EPV_hat']:.4f}")
print(f"VHM hat (σ²_HM)         = {res['VHM_hat']:.4f}")
print(f"k hat                   = {res['k_hat']:.4f}")
print()
print(f"{'Company':>10} {'m_i':>6} {'X̄_i':>8} {'Z_i':>8} {'Pred (unbal)':>14} {'Pred (bal)':>12}")
print("-" * 60)
names = ['A', 'B', 'C']
for i, name in enumerate(names):
    print(f"{name:>10} {res['m_i'][i]:>6.0f} {res['X_bar_i'][i]:>8.4f} {res['Z_i'][i]:>8.4f} "
          f"{res['predictions'][i]:>14.4f} {res['predictions_balanced'][i]:>12.4f}")

print()
# Verify balancing
total_obs = (res['m_i'] * res['X_bar_i']).sum()
total_pred_unbal = (res['m_i'] * res['predictions']).sum()
total_pred_bal   = (res['m_i'] * res['predictions_balanced']).sum()
print(f"Total observed claims (at historical exposure): {total_obs:.2f}")
print(f"Total predicted (unbalanced):                   {total_pred_unbal:.4f}")
print(f"Total predicted (balanced):                     {total_pred_bal:.4f}")
print(f"Balancing complement µ̂_X = {res['mu_hat']:.4f}")

In [ ]:
# ── Example 9.2: Health Insurance — Bühlmann model (equal exposures within group)
# Single period per company; use Bühlmann (not Bühlmann-Straub)
# X_ij = claim amounts per employee; n_i = number of employees (= exposure here)

# For Bühlmann model with equal mij=1, use corollary: m_ij=1 for each observation
# Reconstruct as if we have n_i individual employees with that mean and variance

# Given: mean and std dev of claim amounts per employee
data_ex92 = pd.DataFrame({
    'Company': ['A', 'B', 'C'],
    'n_employees': [350, 673, 979],
    'mean_claim': [467.20, 328.45, 390.23],
    'std_claim':  [116.48, 137.80,  86.50],
})

n_i   = data_ex92['n_employees'].values
mu_i  = data_ex92['mean_claim'].values
sd_i  = data_ex92['std_claim'].values
n_tot = n_i.sum()
r     = len(n_i)

# Overall mean (weighted)
X_bar = (n_i * mu_i).sum() / n_tot

# EPV: E[within-group variance] = sum of (n_i-1)*s_i^2 / sum(n_i - 1)
EPV_hat = ((n_i - 1) * sd_i**2).sum() / (n_i - 1).sum()

# VHM: between-group variance corrected for EPV
between_ss = (n_i * (mu_i - X_bar)**2).sum()
denom_VHM  = n_tot - (n_i**2).sum() / n_tot
VHM_hat    = (between_ss - (r - 1) * EPV_hat) / denom_VHM

k_hat = EPV_hat / VHM_hat
Z_A   = n_i[0] / (n_i[0] + k_hat)   # Company A

# Prediction for Company A with 380 employees next period
pred_per_employee = Z_A * mu_i[0] + (1 - Z_A) * X_bar
pred_total = 380 * pred_per_employee

print("Example 9.2: Health Insurance Bühlmann Credibility")
print("=" * 50)
print(data_ex92.to_string(index=False))
print()
print(f"Overall mean X̄       : {X_bar:.4f}")
print(f"EPV hat               : {EPV_hat:.2f}")
print(f"VHM hat               : {VHM_hat:.2f}")
print(f"k hat                 : {k_hat:.4f}")
print(f"Z_A (n=350)           : {Z_A:.4f}")
print(f"\nPredicted claim per employee (Company A, next period): {pred_per_employee:.4f}")
print(f"Predicted total claim for 380 employees:                {pred_total:,.2f}")

### Semiparametric and Parametric Estimation

**Semiparametric:** Assume a form for the likelihood $f_{X|\Theta}$ but leave the
prior unspecified. For example, if $X_{ij} \sim \text{Poisson}(\lambda_i)$, then
$\mu_{PV} = E[\lambda] = E(X)$, so $\hat{\mu}_{PV} = \bar{X}$.

**Parametric:** Fully specify both likelihood and prior; estimate all hyperparameters
via MLE. More efficient if the model is correct, but sensitive to misspecification.

**Nonparametric** (the main approach above) makes no distributional assumptions
beyond independence and identical distribution within groups — most general.

---
## Summary: Credibility Comparison

| | Classical | Bühlmann | Bayesian |
|---|---|---|---|
| **Framework** | Frequentist | MSE-optimal linear | Full Bayesian |
| **Updating formula** | $Z D + (1-Z)M$ | $Z \bar{X} + (1-Z)\mu_X$ | Posterior mean |
| **$Z$ determination** | Full/partial credibility standard | $Z = n/(n+k)$ | Exact (posterior) |
| **Key assumptions** | Poisson frequency, normal approx | Linear predictor | Prior distribution specified |
| **Strengths** | Simple, intuitive | Rigorous, flexible | Exact when model correct |
| **Weaknesses** | Arbitrary $k$, $\alpha$ | Approximate if non-linear | Prior hard to specify |

**Connection:** Bühlmann = Bayesian when likelihood is LEF and prior is natural conjugate.
**Empirical Bayes:** Estimate EPV and VHM from data (multiple risk groups) → allows Bühlmann–Straub to be applied without assuming a prior distribution.